In [1]:
import ee

# Initialise the Earth Engine connection using your registered project.
# This must be called before any ee.* operations.
ee.Initialize(project='mineral-prospectivity-zim')

print('GEE initialised successfully.')

GEE initialised successfully.


In [2]:
# Bounding box: [west, south, east, north] in decimal degrees
# Covers the Zimbabwe with surrounding context
ROI = ee.Geometry.Rectangle([25.24, -22.42, 33.07, -15.61])

print('Region of interest defined.')
print(f'Bounds: west=28.5, south=-22.5, east=30.5, north=-20.0')

Region of interest defined.
Bounds: west=28.5, south=-22.5, east=30.5, north=-20.0


In [3]:
# Dry season date range — May to October, across five years
START_DATE = '2019-05-01'
END_DATE   = '2024-10-31'
DRY_MONTHS = [5, 6, 7, 8, 9, 10]  # May through October

# Bands to keep — SWIR-heavy selection for geological mapping
BANDS = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'ST_B10']

# Human-readable names for the bands (assigned after selection)
BAND_NAMES = ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2', 'TIR']

print(f'Date range : {START_DATE} to {END_DATE}')
print(f'Dry months : {DRY_MONTHS}')
print(f'Bands      : {BANDS}')

Date range : 2019-05-01 to 2024-10-31
Dry months : [5, 6, 7, 8, 9, 10]
Bands      : ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'ST_B10']


In [4]:
def mask_clouds(image):
    """
    Mask clouds, cloud shadows, and cirrus using Landsat QA_PIXEL,
    combining bit flags with confidence thresholding.
    SR and thermal bands are scaled separately per USGS formulas.

    Reference: USGS, "Landsat Collection 2 Quality Assessment Bands"
    https://www.usgs.gov/landsat-missions/landsat-collection-2-quality-assessment-bands
    """
    qa = image.select('QA_PIXEL')

    dilated_cloud = qa.bitwiseAnd(1 << 1).eq(0)   # Bit 1: Dilated Cloud
    cirrus        = qa.bitwiseAnd(1 << 2).eq(0)   # Bit 2: Cirrus
    cloud         = qa.bitwiseAnd(1 << 3).eq(0)   # Bit 3: Cloud
    cloud_shadow  = qa.bitwiseAnd(1 << 4).eq(0)   # Bit 4: Cloud Shadow

    cloud_confidence  = qa.rightShift(8).bitwiseAnd(3)   # Bits 8-9
    low_cloud_conf    = cloud_confidence.lt(2)

    shadow_confidence = qa.rightShift(10).bitwiseAnd(3)  # Bits 10-11
    low_shadow_conf    = shadow_confidence.lt(2)

    clear_mask = (dilated_cloud.And(cirrus)
                  .And(cloud).And(cloud_shadow)
                  .And(low_cloud_conf).And(low_shadow_conf))

    # SR (optical) bands use one scale/offset; thermal uses a different one — must be split
    SR_BANDS = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']
    sr = image.select(SR_BANDS).multiply(0.0000275).add(-0.2)
    tir = image.select('ST_B10').multiply(0.00341802).add(149.0)

    scaled = image.addBands(sr, None, True).addBands(tir, None, True)

    return (scaled.updateMask(clear_mask)
                  .select(BANDS)
                  .rename(BAND_NAMES)
                  .copyProperties(image, ['system:time_start']))

In [5]:
def prepare_collection(collection_id):
    """
    Load a Landsat collection, filter to our ROI and dry season,
    apply cloud masking, and select our bands.
    """
    return (ee.ImageCollection(collection_id)
          .filterBounds(ROI)
          .filterDate(START_DATE, END_DATE)
          .filter(ee.Filter.calendarRange(DRY_MONTHS[0], DRY_MONTHS[-1], 'month'))
          .map(mask_clouds))

# Load Landsat 8 and 9 Surface Reflectance Tier 1 collections
landsat8 = prepare_collection('LANDSAT/LC08/C02/T1_L2')
landsat9 = prepare_collection('LANDSAT/LC09/C02/T1_L2')

# Merge both collections into one
combined = landsat8.merge(landsat9)

n8 = landsat8.size().getInfo()
n9 = landsat9.size().getInfo()
print(f'Landsat 8 scenes : {n8}')
print(f'Landsat 9 scenes : {n9}')
print(f'Total scenes     : {n8 + n9}')


Landsat 8 scenes : 2607
Landsat 9 scenes : 1304
Total scenes     : 3911


In [6]:
# Reduce the collection to a single image using the median
composite = combined.median().clip(ROI)

# Confirm the output band names
print('Composite bands:', composite.bandNames().getInfo())

Composite bands: ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2', 'TIR']


In [7]:
task = ee.batch.Export.image.toAsset(
    image       = composite,
    description = 'landsat_Zimbabwe_dry_season',
    assetId     = 'projects/mineral-prospectivity-zim/assets/landsat_Zimbabwe_dry_season',
    region      = ROI,
    scale       = 30,
    crs         = 'EPSG:4326',
    maxPixels   = 1e9
)

task.start()

print('Export task submitted successfully.')
print()
print('Next steps:')
print('  1. Go to https://code.earthengine.google.com → Tasks tab')
print('  2. Wait for "landsat_gwanda_dry_season" to show COMPLETED')
print('  3. Download the GeoTIFF from your Google Drive')
print('  4. Place it in: data/raw/landsat/')

Export task submitted successfully.

Next steps:
  1. Go to https://code.earthengine.google.com → Tasks tab
  2. Wait for "landsat_gwanda_dry_season" to show COMPLETED
  3. Download the GeoTIFF from your Google Drive
  4. Place it in: data/raw/landsat/


In [8]:
# GEE objects live on Google's servers, not in local memory,
# so there is nothing to del. We just release the local Python references.
del landsat8, landsat9, combined, composite, task

import gc
gc.collect()

print('Done. Monitor the export task at https://code.earthengine.google.com')

Done. Monitor the export task at https://code.earthengine.google.com
